# Sistem Rekomendasi Item Menggunakan User-Based KNN

**Dataset**: Retailrocket (events_P1.csv dan events_P2.csv)  
**Tujuan**: memilih user secara acak tetapi dapat direproduksi, mencari 5 tetangga terdekat (KNN, \(k=5\)), menampilkan 3 rekomendasi teratas, lalu mengevaluasi daftar rekomendasi dengan Hit@5 dan NDCG@5.

Import Library

In [40]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

print("Pustaka berhasil diimpor.")

Pustaka berhasil diimpor.


Menentukan Lokasi File

In [41]:
BASE_DIR = Path(r"D:\ANNAS\DOKUMEN\TUGAS KULIAH\SEMESTER 7\GRAF SAINS DATA\UTS")

P1_PATH = BASE_DIR / "events_P1.csv"
P2_PATH = BASE_DIR / "events_P2.csv"

print("P1 ditemukan:", P1_PATH.exists(), "-", P1_PATH)
print("P2 ditemukan:", P2_PATH.exists(), "-", P2_PATH)

if not P1_PATH.exists() or not P2_PATH.exists():
    raise FileNotFoundError(
        "File P1/P2 tidak ditemukan. Periksa kembali BASE_DIR."
    )

P1 ditemukan: True - D:\ANNAS\DOKUMEN\TUGAS KULIAH\SEMESTER 7\GRAF SAINS DATA\UTS\events_P1.csv
P2 ditemukan: True - D:\ANNAS\DOKUMEN\TUGAS KULIAH\SEMESTER 7\GRAF SAINS DATA\UTS\events_P2.csv


Baca Data dan Periksa Struktur Data

In [42]:
p1 = pd.read_csv(P1_PATH)
p2 = pd.read_csv(P2_PATH)

print("Ukuran P1:", p1.shape)
print("Ukuran P2:", p2.shape)

print("\nKolom P1:")
print(p1.columns.tolist())

display(p1.head())
display(p2.head())

Ukuran P1: (1655156, 5)
Ukuran P2: (367315, 5)

Kolom P1:
['timestamp', 'visitorid', 'event', 'itemid', 'transactionid']


,timestamp,visitorid,event,itemid,transactionid
0,1433301556321,869738,view,7415,NaN
1,1433302935771,101612,view,293934,NaN
2,1433300277957,845514,view,214957,NaN
3,1433299970586,170832,view,42002,NaN
4,1433302407102,295384,view,88086,NaN


,timestamp,visitorid,event,itemid,transactionid
0,1440297055552,32427,view,414410,NaN
1,1440298060706,755907,view,79239,NaN
2,1440297618054,252831,view,40502,NaN
3,1440294155928,1256758,view,160637,NaN
4,1440298691004,1275666,view,185620,NaN


Validasi Kolom dan Kualitas Data

In [43]:
required_cols = {"visitorid", "itemid"}
missing_p1 = required_cols - set(p1.columns)
missing_p2 = required_cols - set(p2.columns)

if missing_p1 or missing_p2:
    raise ValueError(
        f"Kolom wajib tidak ditemukan. P1: {missing_p1}; P2: {missing_p2}"
    )

for name, df in [("P1", p1), ("P2", p2)]:
    print(f"\n=== {name} ===")
    print("Baris:", len(df))
    print("visitorid kosong:", df["visitorid"].isna().sum())
    print("itemid kosong:", df["itemid"].isna().sum())
    if "event" in df.columns:
        display(df["event"].value_counts(dropna=False).rename_axis("event").to_frame("jumlah"))

# Buang baris tanpa visitorid/itemid karena tidak dapat membentuk interaksi
p1 = p1.dropna(subset=["visitorid", "itemid"]).copy()
p2 = p2.dropna(subset=["visitorid", "itemid"]).copy()

print("\nUkuran setelah menghapus ID kosong:")
print("P1:", p1.shape)
print("P2:", p2.shape)


=== P1 ===
Baris: 1655156
visitorid kosong: 0
itemid kosong: 0


,jumlah
event,
view,1599653
addtocart,41835
transaction,13668



=== P2 ===
Baris: 367315
visitorid kosong: 0
itemid kosong: 0


,jumlah
event,
view,354622
addtocart,9620
transaction,3073



Ukuran setelah menghapus ID kosong:
P1: (1655156, 5)
P2: (367315, 5)


Bentuk Interaksi User-Item Biner dari P1

In [44]:
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder

interaksi_p1 = p1[["visitorid", "itemid"]].drop_duplicates().copy()

# Encode ID asli menjadi indeks numerik untuk matriks sparse
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

interaksi_p1["user_idx"] = user_encoder.fit_transform(interaksi_p1["visitorid"])
interaksi_p1["item_idx"] = item_encoder.fit_transform(interaksi_p1["itemid"])

n_users = len(user_encoder.classes_)
n_items = len(item_encoder.classes_)

# Matriks user-item sparse biner: hemat memori dibanding matriks padat
user_item = csr_matrix(
    (
        np.ones(len(interaksi_p1), dtype=np.float32),
        (interaksi_p1["user_idx"].to_numpy(), interaksi_p1["item_idx"].to_numpy())
    ),
    shape=(n_users, n_items),
    dtype=np.float32
)

# Buat peta ID asli ke indeks matriks
user_to_idx = {user_id: idx for idx, user_id in enumerate(user_encoder.classes_)}
item_to_idx = {item_id: idx for idx, item_id in enumerate(item_encoder.classes_)}

print("Jumlah pasangan user-item unik P1:", len(interaksi_p1))
print("Dimensi matriks user-item:", user_item.shape)
print("Jumlah user unik P1:", n_users)
print("Jumlah item unik P1:", n_items)
print("Jumlah interaksi tersimpan dalam matriks:", user_item.nnz)

# Tampilkan cuplikan kecil dalam bentuk tabel agar tidak mengubah matriks besar menjadi dense
preview_users = min(5, n_users)
preview_items = min(10, n_items)
preview = pd.DataFrame(
    user_item[:preview_users, :preview_items].toarray(),
    columns=item_encoder.classes_[:preview_items],
    index=user_encoder.classes_[:preview_users]
)
display(preview)

Jumlah pasangan user-item unik P1: 1291947
Dimensi matriks user-item: (861498, 189511)
Jumlah user unik P1: 861498
Jumlah item unik P1: 189511
Jumlah interaksi tersimpan dalam matriks: 1291947


,3,4,6,9,15,16,17,19,25,26
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Pilih User acak dan dapat direproduksi

In [45]:
RANDOM_SEED = 100

users_p2 = set(p2["visitorid"].dropna().unique())
candidate_users = [u for u in user_encoder.classes_ if u in users_p2]

if len(candidate_users) == 0:
    raise ValueError(
        "Tidak ada user yang muncul pada P1 dan P2. Periksa pembagian periode atau tipe ID."
    )

rng = np.random.default_rng(RANDOM_SEED)
target_user = rng.choice(np.asarray(candidate_users))

print("Random seed:", RANDOM_SEED)
print("Jumlah kandidat user (muncul di P1 dan P2):", len(candidate_users))
print("User target terpilih:", target_user)

Random seed: 100
Jumlah kandidat user (muncul di P1 dan P2): 16802
User target terpilih: 1088457


Hitung kemiripan user dan cari 5 tetangga terdekat

In [46]:
K = 5

if n_users < K + 1:
    raise ValueError("Jumlah user terlalu sedikit untuk memilih 5 tetangga.")

target_idx = user_to_idx[target_user]
target_vector = user_item[target_idx]

# Cosine similarity sparse terhadap seluruh user; hanya satu baris hasil
similarities = cosine_similarity(target_vector, user_item).ravel()

similarity_df = pd.DataFrame({
    "visitorid": user_encoder.classes_,
    "similarity": similarities
})

# Hilangkan target user sendiri
similarity_df = similarity_df[
    similarity_df["visitorid"] != target_user
]

knn = similarity_df.sort_values(
    ["similarity", "visitorid"],
    ascending=[False, True]
).head(K).reset_index(drop=True)

print("User target:", target_user)
print(f"5 tetangga terdekat (K={K}):")
display(knn)

User target: 1088457
5 tetangga terdekat (K=5):


,visitorid,similarity
0,115947,0.707107
1,744148,0.707107
2,1085304,0.707107
3,1176088,0.707107
4,1191780,0.707107


Buat skor kandidat item dari 5 tetangga

In [47]:
# Interaksi target yang sudah diketahui dari P1
target_rows = interaksi_p1[interaksi_p1["visitorid"] == target_user]
items_seen_by_target = set(target_rows["itemid"])

# Ambil indeks matriks untuk 5 tetangga
neighbor_ids = knn["visitorid"].tolist()
neighbor_indices = [user_to_idx[user_id] for user_id in neighbor_ids]
neighbor_weights = knn["similarity"].to_numpy(dtype=np.float32)

# Skor setiap item = jumlah berbobot similarity tetangga yang berinteraksi dengannya
neighbor_matrix = user_item[neighbor_indices]
item_scores = np.asarray(neighbor_matrix.T.dot(neighbor_weights)).ravel()

recommendations = pd.DataFrame({
    "itemid": item_encoder.classes_,
    "score": item_scores
})

recommendations = recommendations[
    ~recommendations["itemid"].isin(items_seen_by_target)
]

recommendations = recommendations.sort_values(
    ["score", "itemid"],
    ascending=[False, True]
).reset_index(drop=True)

top5 = recommendations.head(5).copy()
top3 = recommendations.head(3).copy()

print("Jumlah item yang sudah pernah disentuh target di P1:", len(items_seen_by_target))
print("\nTop-3 item rekomendasi:")
display(top3)

print("\nTop-5 item untuk evaluasi:")
display(top5)

Jumlah item yang sudah pernah disentuh target di P1: 1

Top-3 item rekomendasi:


,itemid,score
0,39449,2.828427
1,419060,0.707107
2,3,0.000000



Top-5 item untuk evaluasi:


,itemid,score
0,39449,2.828427
1,419060,0.707107
2,3,0.000000
3,4,0.000000
4,6,0.000000


Tentukan Item Relevan dari P2

In [48]:
relevant_items = set(
    p2.loc[p2["visitorid"] == target_user, "itemid"].unique()
)

recommended_top5 = top5["itemid"].tolist()
recommended_top3 = top3["itemid"].tolist()

print("Jumlah item relevan user target pada P2:", len(relevant_items))
print("Item relevan P2 (maksimal 20 pertama):", list(relevant_items)[:20])
print("Rekomendasi top-5:", recommended_top5)
print("Rekomendasi top-3:", recommended_top3)

Jumlah item relevan user target pada P2: 1
Item relevan P2 (maksimal 20 pertama): [np.int64(372102)]
Rekomendasi top-5: [39449, 419060, 3, 4, 6]
Rekomendasi top-3: [39449, 419060, 3]


Hitung Hit@5

In [49]:
hits = [1 if item in relevant_items else 0 for item in recommended_top5]

hit_at_5 = int(any(hits))

print("Indikator relevansi pada top-5:", hits)
print("Hit@5:", hit_at_5)

Indikator relevansi pada top-5: [0, 0, 0, 0, 0]
Hit@5: 0


Hitung NDCG@5

In [50]:
def dcg_at_k(relevance, k):
    relevance = np.asarray(relevance, dtype=float)[:k]
    if relevance.size == 0:
        return 0.0
    discounts = np.log2(np.arange(2, relevance.size + 2))
    return float(np.sum(relevance / discounts))

relevance_top5 = np.array([
    1 if item in relevant_items else 0
    for item in recommended_top5
], dtype=int)

if len(relevant_items) == 0:
    ndcg_at_5 = np.nan
    print("NDCG@5 tidak dapat dihitung: user target tidak memiliki item relevan di P2.")
else:
    dcg = dcg_at_k(relevance_top5, 5)

    ideal_length = min(len(relevant_items), 5)
    ideal_relevance = np.ones(ideal_length, dtype=int)
    idcg = dcg_at_k(ideal_relevance, 5)

    ndcg_at_5 = dcg / idcg if idcg > 0 else np.nan

    print("Relevansi top-5:", relevance_top5.tolist())
    print("DCG@5:", round(dcg, 6))
    print("IDCG@5:", round(idcg, 6))
    print("NDCG@5:", round(ndcg_at_5, 6) if pd.notna(ndcg_at_5) else ndcg_at_5)

Relevansi top-5: [0, 0, 0, 0, 0]
DCG@5: 0.0
IDCG@5: 1.0
NDCG@5: 0.0


Ringkasan Hasil

In [51]:
print("=" * 60)
print("RINGKASAN EKSPERIMEN USER-BASED KNN")
print("=" * 60)
print(f"Random seed       : {RANDOM_SEED}")
print(f"User target       : {target_user}")
print(f"Jumlah tetangga   : {K}")

print("\n5 tetangga terdekat:")
display(knn)

print("3 item rekomendasi utama:")
display(top3)

print("Daftar relevansi top-5:", relevance_top5.tolist())
print(f"Hit@5             : {hit_at_5}")
print(
    "NDCG@5            : "
    + (f"{ndcg_at_5:.6f}" if pd.notna(ndcg_at_5) else "Tidak terdefinisi")
)

RINGKASAN EKSPERIMEN USER-BASED KNN
Random seed       : 100
User target       : 1088457
Jumlah tetangga   : 5

5 tetangga terdekat:


,visitorid,similarity
0,115947,0.707107
1,744148,0.707107
2,1085304,0.707107
3,1176088,0.707107
4,1191780,0.707107


3 item rekomendasi utama:


,itemid,score
0,39449,2.828427
1,419060,0.707107
2,3,0.000000


Daftar relevansi top-5: [0, 0, 0, 0, 0]
Hit@5             : 0
NDCG@5            : 0.000000
